In [ ]:
# ==============================================================================
# Reprojecting Multispectral Orthomosaic to BNG
# Transformation: OSTN15 (Horizontal)
# From: EPSG:32630 (UTM 30N)
# To:   EPSG:27700 (BNG)
# Date: April 2026
# ==============================================================================

In [ ]:
%pip install pyproj rasterio -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
## Test if the OSTN15 horizontal grid is working
import os
import pyproj
from pyproj import Transformer

## Setup Paths
# Local Users: Leave it as BASE_DIR = ""
# Colab Users: Use your Drive path below (i.e BASE_DIR = "/content/drive/MyDrive/Folder_Name/")

BASE_DIR = "/content/drive/MyDrive/Folder_Name/"

# 1. Point PyProj to your local/Drive grids folder
grid_folder = os.path.join(BASE_DIR, "Transformation_Grids")  # Replace "Transformation_Grids" with the name of the folder you kept the grid files
pyproj.datadir.append_data_dir(grid_folder)

# 2. Build the 2D-only pipeline (Planimetric shift)
t = Transformer.from_pipeline(
    "+proj=pipeline "
    "+step +inv +proj=utm +zone=30 +ellps=WGS84 "
    "+step +proj=hgridshift +grids=ostn15_etrs_to_osgb.gsb "
    "+step +proj=tmerc +lat_0=49 +lon_0=-2 +k=0.9996012717 "
    "+x_0=400000 +y_0=-100000 +ellps=airy +units=m +no_defs"
)

# 3. Transform a test point (UTM Easting, UTM Northing)
e_bng, n_bng = t.transform(456000.0, 5850000.0)

print("Validation Completed!!!")
print(f"BNG Easting:  {e_bng:.3f}")
print(f"BNG Northing: {n_bng:.3f}")

In [ ]:
import os
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import pyproj

# 1. DEFINE THE FUNCTION
def reproject_multi_bng(input_path, output_path, grid_folder):
    """
    Reprojects multi-band rasters to BNG using OSTN15 for horizontal precision.
    """
    # Tell PyProj where to find custom grids
    pyproj.datadir.append_data_dir(grid_folder)

    # Anchor the grid path to an absolute location for Rasterio/GDAL
    gsb_path = os.path.abspath(os.path.join(grid_folder, "ostn15_etrs_to_osgb.gsb"))

    # Define the high-precision BNG pipeline
    bng_base = "+proj=tmerc +lat_0=49 +lon_0=-2 +k=0.9996012717 +x_0=400000 +y_0=-100000 +ellps=airy +units=m +no_defs"
    bng_2d_pipeline = f"{bng_base} +nadgrids=@{gsb_path}"

    with rasterio.open(input_path) as src:
        # Calculate the transformation math
        transform, width, height = calculate_default_transform(
            "EPSG:32630", "EPSG:27700", src.width, src.height, *src.bounds
        )

        kwargs = src.meta.copy()
        kwargs.update({
            'crs': 'EPSG:27700',
            'transform': transform,
            'width': width,
            'height': height
        })

        with rasterio.open(output_path, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                print(f"Processing Band {i} of {src.count}...")
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs="EPSG:32630",
                    dst_transform=transform,
                    dst_crs=bng_2d_pipeline,
                    resampling=Resampling.nearest
                )
    print(f"✅ Success: {output_path}")

# 2. EXECUTE THE SCRIPT
if __name__ == "__main__":

## Setup Paths
# Local Users: Leave it as BASE_DIR = ""
# Colab Users: Use your Drive path below (i.e BASE_DIR = "/content/drive/MyDrive/Folder_Name/")

    BASE_DIR = "/content/drive/MyDrive/Folder_Name/"

    input_file = os.path.join(BASE_DIR, 'Multispectral_Ortho.tif') # Replace "Multispectral_Ortho.tif" with your input file name
    output_file = os.path.join(BASE_DIR, 'Multispectral_Ortho_BNG.tif')  # Replace "Multispectral_Ortho_BNG.tif" with your preferred output file name
    grid_dir = os.path.join(BASE_DIR, 'Transformation_Grids') # Replace "Transformation_Grids" with the name of the folder you kept the grid files

    # Run the function
    reproject_multi_bng(input_file, output_file, grid_dir)